# Geometry certification

Before a long science run, check that the sampling geometry is healthy. Then declare near-linear axes `identically_linear` and check again.

Rebuild the decentered Discovery model here (do not `%run` notebook 3). AEI-DR2 combined J1022+1001; expand at the prior center because this pulsar’s PX is negative.


In [ ]:
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")

from pathlib import Path
import discovery as ds
from metapulsar import create_metapulsar
from nltiming import TimingSpec, TimingExpansionSpec, certify_decentered_geometry
from nltiming.sampling import numpyro

numpyro.ensure_x64()

DATA = Path("..") / "data" / "J1022+1001"
pulsar = create_metapulsar(
    {"combined": [{
        "par": DATA / "J1022+1001.par",
        "tim": DATA / "J1022+1001.tim",
        "timing_package": "tempo2",
    }]},
    combination_strategy="per_pta",
    use_pulse_numbers="reuse",
)
nd = {f"{pulsar.name}_efac": 1.0, f"{pulsar.name}_log10_t2equad": -8.0}

spec = TimingSpec(
    engines="jug", name="timing",
    expansion=TimingExpansionSpec.prior_center(),
)
timing = spec.for_pulsar(pulsar)
likelihood = ds.PulsarLikelihood([
    pulsar.residuals,
    ds.makenoise_measurement_simple(pulsar, nd),
    *timing.discovery_signals(),
])
model = numpyro.decentered_model(likelihood, timing, fixed=nd)


## Certify the default geometry


In [ ]:
report = certify_decentered_geometry(model, timing, hyper_points=[{}])
print(report.passed, report)


## Declare linear axes and certify again

Union the auto-derived set with spin and ecliptic position so DM/jumps stay certified.


In [ ]:
spec_lin = TimingSpec(
    engines="jug",
    identically_linear=sorted(set(timing.identically_linear) | {"F0", "F1", "ELONG", "ELAT"}),
    name="timing",
    expansion=TimingExpansionSpec.prior_center(),
)
timing_lin = spec_lin.for_pulsar(pulsar)
model_lin = numpyro.decentered_model(
    ds.PulsarLikelihood([
        pulsar.residuals,
        ds.makenoise_measurement_simple(pulsar, nd),
        *timing_lin.discovery_signals(),
    ]),
    timing_lin,
    fixed=nd,
)
report_lin = certify_decentered_geometry(model_lin, timing_lin, hyper_points=[{}])
print(timing_lin.identically_linear)
print(report.passed, report_lin.passed)
print(report, report_lin)
